# From a small neural network to a deeper one — PyTorch

The network grows *during training*, and we print its exact architecture, parameter count, and growth reason. The guaranteed growth switch is only a demonstration; plateau mode uses validation data.


## 1. Import our inspectable neural network and the policy
`nn.Linear` owns the neurons, weight tensors and biases; `nn.ModuleList` holds blocks we add at runtime.

In [ ]:
import sys
from pathlib import Path
root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root / "src"))
from adaptive_nn import AdaptiveNetwork, ResidualBlock, ScalePolicy, GrowthController
import torch
import matplotlib.pyplot as plt
torch.manual_seed(7)


## 2. Start small; count actual trainable parameters

In [ ]:
net = AdaptiveNetwork(inputs=4, outputs=1, width=16)
x = torch.randn(5, 4)
print(net)
print("Initial depth:", net.depth, "parameters:", net.parameter_count)
print("Prediction shape:", net(x).shape)

## 3. Physically add a trainable hidden block
Zero-initializing the new residual branch makes predictions identical at insertion. The extra block is real and will learn with gradient descent.

In [ ]:
original = net(x).detach().clone()
net.grow()
print(net)
print("New depth:", net.depth, "parameters:", net.parameter_count)
print("Maximum output change right at insertion:", (original - net(x)).abs().max().item())

## 4. Run the complete training + scaling demonstration
The runner writes a reproducible CSV and JSON log. We force one growth at epoch 15 so the transformation always appears on screen. Remove that flag to test the validation-plateau heuristic.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, str(root / "train.py"), "--epochs", "90", "--force-grow-epoch", "15", "--output", str(root / "outputs")], check=True)

## 5. Inspect training loss, architecture depth, and decisions

In [ ]:
import csv, json
with (root / "outputs" / "history.csv").open() as f:
    rows = list(csv.DictReader(f))
epoch = [int(r["epoch"]) for r in rows]
loss = [float(r["val_loss"]) for r in rows]
depth = [int(r["depth"]) for r in rows]
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axes[0].plot(epoch, loss, label="held-out MSE")
axes[0].set_ylabel("Validation loss")
axes[0].legend()
axes[1].step(epoch, depth, where="post")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Hidden stages")
plt.tight_layout()
plt.show()
print(json.dumps(json.loads((root / "outputs" / "growth_events.json").read_text()), indent=2))

## 6. Engineering distinction
Architecture scaling adds **parameters and blocks**. Runtime scaling (larger batches, more GPUs, Kubernetes replicas) is different. A larger network may **not** generalize better, so validation metrics and cost must justify growth.